# 04 - Inference Benchmarking
Measure the forward-pass inference time of the trained models to benchmark computational cost vs accuracy.

In [1]:
import sys
import time
import torch
import pandas as pd
from torch_geometric.loader import DataLoader
import numpy as np

sys.path.append("..")
from src.dataset import MDTrajectoryDataset
from src.models.ace_wrapper import ACEWrapper
from src.models.mace_wrapper import MACEWrapper

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Benchmarking on device: {device}")

Benchmarking on device: cuda


In [2]:
# Load the Test Dataset
# Using batch_size=1 to simulate sequential MD steps
test_ds = MDTrajectoryDataset("../data/test.extxyz", cutoff=5.0)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).


In [3]:
def benchmark_inference(model, loader, device, num_warmup=10):
    model.eval()
    model = model.to(device)
    
    times = []
    
    for i, batch in enumerate(loader):
        batch = batch.to(device)
        
        if device == "cuda":
            torch.cuda.synchronize()
        
        start_time = time.time()
        
        # Forward pass (forces require grad)
        _ = model(batch)
        
        if device == "cuda":
            torch.cuda.synchronize()
            
        end_time = time.time()
        
        if i >= num_warmup:
            times.append((end_time - start_time) * 1000) # milliseconds
            
    return np.mean(times), np.std(times)

In [4]:
# 1. Benchmark ACE (Local)
ace_model = ACEWrapper(
    num_elements=120,
    num_radial=8,
    l_max=2,
    r_cut=5.0,
    hidden_dim=32
)
try:
    ace_model.load_state_dict(torch.load("../data/ace_model_100.pth", map_location=device))
    print("Benchmarking ACE...")
    ace_mean, ace_std = benchmark_inference(ace_model, test_loader, device)
    print(f"ACE Inference: {ace_mean:.2f} ± {ace_std:.2f} ms/step")
except FileNotFoundError:
    print("ace_model_100.pth not found. Train the model first.")
    ace_mean, ace_std = float('nan'), float('nan')

# 2. Benchmark MACE (Message Passing)
mace_model = MACEWrapper(
    num_elements=120,
    r_cut=5.0,
    num_radial=8,
    l_max=2,
    num_blocks=2,
    node_dim=16
)
try:
    mace_model.load_state_dict(torch.load("../data/mace_model_100.pth", map_location=device))
    print("Benchmarking MACE...")
    mace_mean, mace_std = benchmark_inference(mace_model, test_loader, device)
    print(f"MACE Inference: {mace_mean:.2f} ± {mace_std:.2f} ms/step")
except FileNotFoundError:
    print("mace_model_100.pth not found. Train the model first.")
    mace_mean, mace_std = float('nan'), float('nan')

# 3. Save Results
results = pd.DataFrame({
    "Model": ["ACE", "MACE"],
    "Inference_Time_ms": [ace_mean, mace_mean],
    "Inference_Std_ms": [ace_std, mace_std]
})
results.to_csv("../data/inference_metrics.csv", index=False)
print("Saved inference metrics to data/inference_metrics.csv")

Benchmarking ACE...
ACE Inference: 9.91 ± 1.77 ms/step
Benchmarking MACE...
MACE Inference: 59.60 ± 3.77 ms/step
Saved inference metrics to data/inference_metrics.csv
